# Voice Cloning & Multilingual Speech with Voxtral TTS

<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/tts/voxtral_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Author:** Utkarsh Kanwat ([@ukanwat](https://github.com/ukanwat)) — Community Contributor

[Voxtral TTS](https://docs.mistral.ai/studio-api/audio/text_to_speech) is Mistral's text-to-speech model. It turns text into natural, expressive speech, supports nine languages, and can clone a voice from a few seconds of reference audio — all through a single API.

This notebook is a practical tour of the **text-to-speech** side of the Voxtral family. (Audio *understanding* and transcription are covered separately.) You will:

1. Generate speech with built-in **preset voices**.
2. **Clone a voice** zero-shot from a short reference clip.
3. Make a voice speak in **other languages** (cross-lingual transfer).
4. Use **streaming** output and compare audio formats and latency.
5. Wire TTS into a small **voice-reply agent** (LLM → speech).
6. *(Appendix)* Run the **open-weights** `Voxtral-4B-TTS` model locally.

Everything runs against the hosted API with a `MISTRAL_API_KEY`, and the notebook is self-contained — no audio files need to be downloaded.

| | |
|---|---|
| Model | `voxtral-mini-tts-2603` |
| Endpoint | `/v1/audio/speech` |
| Pricing | ~\$0.016 / 1k characters |
| Open weights | [`mistralai/Voxtral-4B-TTS-2603`](https://huggingface.co/mistralai/Voxtral-4B-TTS-2603) (Apache-2.0) |

## Setup

Voxtral TTS needs the **`mistralai` 2.x** SDK, which requires **Python ≥ 3.10** (Colab's default runtime is fine). Note the import path: in `mistralai` 2.x the client is imported as `from mistralai.client import Mistral`.

In [ ]:
%pip install -q "mistralai==2.5.0"

### Authentication

You'll need a `MISTRAL_API_KEY` — get one from the [Mistral console](https://console.mistral.ai/). In Colab, the cleanest option is the **Secrets** panel (🔑 in the left sidebar): add a secret named `MISTRAL_API_KEY` and enable it for this notebook. Never hard-code your key into the notebook.

In [ ]:
import os
import base64
import time
from pathlib import Path
from getpass import getpass

api_key = os.environ.get("MISTRAL_API_KEY")

# Colab Secrets panel
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("MISTRAL_API_KEY")
    except Exception:
        pass

# Fallback: secure prompt (input is hidden)
if not api_key:
    api_key = getpass("Enter your MISTRAL_API_KEY: ")

from mistralai.client import Mistral
from IPython.display import Audio, display

client = Mistral(api_key=api_key)
TTS_MODEL = "voxtral-mini-tts-2603"

### Helpers

`client.audio.speech.complete` returns the generated audio as base64 in `audio_data`. These two helpers decode it to a file and return an inline player.

In [ ]:
def synthesize(text, voice_id, response_format="mp3"):
    """Generate speech and return the decoded audio bytes."""
    response = client.audio.speech.complete(
        model=TTS_MODEL,
        input=text,
        voice_id=voice_id,
        response_format=response_format,
    )
    return base64.b64decode(response.audio_data)


def speak(text, voice_id, path, response_format="mp3"):
    """Synthesize `text` with `voice_id`, save to `path`, return a player widget."""
    Path(path).write_bytes(synthesize(text, voice_id, response_format))
    return Audio(filename=path)

## 1. Preset voices

Voxtral TTS ships with built-in preset voices, so you can generate speech with zero setup. Each preset has a human-readable `slug` (e.g. `gb_jane_neutral`) that you pass as `voice_id`.

In [ ]:
speak(
    "Hello! This is Voxtral, Mistral's text-to-speech model, "
    "speaking with one of the built-in preset voices.",
    voice_id="gb_jane_neutral",
    path="preset_voice.mp3",
)

In [ ]:
# Browse the preset-voice catalogue.
voices = client.audio.voices.list(type_="preset")
print(f"{voices.total} preset voices available, for example:")
for v in voices.items[:8]:
    print(f"  {v.slug:24} {v.name}")

## 2. Zero-shot voice cloning

Cloning needs a short reference clip — roughly **5–25 seconds** works best (as little as 3s is accepted). To keep this notebook self-contained, we first *synthesize* a reference clip with a preset voice. **In a real application, replace this with an actual recording** of the speaker you want to clone (see the optional upload cell below).

> **Note:** creating custom (cloned) voices requires an **active paid plan**. Preset voices and everything else in this notebook work on all plans. If cloning isn't available on your account, the cells below automatically fall back to a preset voice so the notebook still runs end to end.

In [ ]:
reference_path = "reference.mp3"
Path(reference_path).write_bytes(
    synthesize(
        "This is a reference recording. In a real project you would supply a short "
        "clip of the voice you want to clone, such as a voice note or a podcast snippet.",
        voice_id="gb_jane_neutral",
    )
)
Audio(filename=reference_path)

In [ ]:
# Create a reusable cloned voice from the reference clip (falls back gracefully).
clone_voice_id = None
try:
    created = client.audio.voices.create(
        name="cookbook-cloned-voice",
        sample_audio=base64.b64encode(Path(reference_path).read_bytes()).decode(),
        sample_filename=reference_path,
        languages=["en"],
    )
    clone_voice_id = created.id
    print("Created cloned voice:", clone_voice_id)
except Exception as e:
    print("Voice cloning unavailable (custom voices require an active paid plan):")
    print(" ", e)
    print("Falling back to a preset voice for the remaining demos.")

# Voice used for the cloning + cross-lingual demos below:
# the cloned voice if we have one, otherwise a preset so the notebook still runs.
demo_voice = clone_voice_id or "gb_jane_neutral"

In [ ]:
# The demo voice (cloned, or preset fallback) reads text it has never seen.
speak(
    "And now the voice reads a brand new sentence, in the timbre of the reference clip.",
    voice_id=demo_voice,
    path="demo_voice.mp3",
)

**Clone your own voice (optional, Colab).** Uncomment to upload a 5–25s clip of yourself and hear it read new text.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # pick a 5-25s .mp3/.wav recording
# fname = next(iter(uploaded))
# created = client.audio.voices.create(
#     name="my-voice",
#     sample_audio=base64.b64encode(Path(fname).read_bytes()).decode(),
#     sample_filename=fname,
#     languages=["en"],
# )
# speak("Hello, this is my own cloned voice.", voice_id=created.id, path="my_voice.mp3")

## 3. Cross-lingual transfer

A single voice can speak multiple languages. Here the same `demo_voice` (your cloned voice, or the preset fallback) reads the same idea in English, French, Spanish, and Hindi.

In [ ]:
phrases = {
    "English": "Voxtral can read this text aloud in a natural voice.",
    "French":  "Voxtral peut lire ce texte à voix haute avec une voix naturelle.",
    "Spanish": "Voxtral puede leer este texto en voz alta con una voz natural.",
    "Hindi":   "वोक्सट्राल इस पाठ को एक स्वाभाविक आवाज़ में पढ़ सकता है।",
}

for language, text in phrases.items():
    print(language)
    display(speak(text, voice_id=demo_voice, path=f"speech_{language}.mp3"))

## 4. Streaming and audio formats

For interactive apps you usually want to start playing audio before the whole clip is generated. Set `stream=True` and consume the `speech.audio.delta` events. Below we measure time-to-first-audio for two formats — `mp3` (compressed) and `pcm` (raw) — and play back the streamed `mp3`.

In [ ]:
def stream_speech(text, voice_id, response_format="mp3"):
    """Stream speech; return (time_to_first_audio_seconds, joined_audio_bytes)."""
    start = time.time()
    first = None
    chunks = []
    with client.audio.speech.complete(
        model=TTS_MODEL,
        input=text,
        voice_id=voice_id,
        response_format=response_format,
        stream=True,
    ) as stream:
        for event in stream:
            if event.event == "speech.audio.delta":
                if first is None:
                    first = time.time() - start
                chunks.append(base64.b64decode(event.data.audio_data))
            elif event.event == "speech.audio.done":
                break
    return first, b"".join(chunks)


text = "Streaming lets you start playback before the whole clip is ready."
for fmt in ["mp3", "pcm"]:
    ttfa, audio = stream_speech(text, "gb_jane_neutral", fmt)
    print(f"{fmt:>3}: time-to-first-audio = {ttfa:.2f}s, total bytes = {len(audio):,}")

In [ ]:
# Save and play the streamed mp3.
_, mp3_bytes = stream_speech(text, "gb_jane_neutral", "mp3")
Path("streamed.mp3").write_bytes(mp3_bytes)
Audio(filename="streamed.mp3")

## 5. A minimal voice-reply agent

Putting it together: an LLM writes a short reply, and Voxtral speaks it. This is the output half of a voice assistant (pair it with Voxtral transcription for the input half).

In [ ]:
user_prompt = "In one or two sentences, why is text-to-speech useful for accessibility?"

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "system", "content": "You are concise. Reply in 1-2 plain sentences, no markdown."},
        {"role": "user", "content": user_prompt},
    ],
)
reply = response.choices[0].message.content
print("Assistant:", reply)

speak(reply, voice_id=demo_voice, path="assistant_reply.mp3")

## Cleanup

Remove the cloned voice if we created one (preset voices need no cleanup).

In [ ]:
if clone_voice_id:
    client.audio.voices.delete(voice_id=clone_voice_id)
    print("Deleted", clone_voice_id)
else:
    print("No custom voice was created — nothing to clean up.")

## Appendix — running the open weights locally

Voxtral TTS is released under Apache-2.0, so you can self-host it for offline or high-volume use. The model is `mistralai/Voxtral-4B-TTS-2603` (~4B parameters). It needs a GPU and **will not run on a free Colab CPU runtime**; a single mid-range GPU is enough.

```bash
pip install vllm
vllm serve mistralai/Voxtral-4B-TTS-2603
```

Once served, it exposes the same OpenAI-compatible `/v1/audio/speech` interface used above. You can point the SDK at your local endpoint with `Mistral(api_key="...", server_url="http://localhost:8000")`. See the [model card](https://huggingface.co/mistralai/Voxtral-4B-TTS-2603) for hardware and quantization options.

## Wrap-up

You generated speech with preset voices, cloned a voice zero-shot, transferred it across languages, streamed audio with latency measurements, and built a small LLM → speech reply loop — then saw how to run the open weights locally.

**Where to go next**

- [Voxtral TTS documentation](https://docs.mistral.ai/studio-api/audio/text_to_speech)
- [Audio understanding & transcription](https://docs.mistral.ai/capabilities/audio)
- [Agents API](https://docs.mistral.ai/studio-api/agents/introduction) — combine transcription + reasoning + TTS into a full voice agent